# Corticall - Gate 0: in-distribution validation (GO / NO-GO)

Run on **Kaggle** with a **GPU T4** and **Internet = ON**. Recipe (from the smoke test): **Run All -> Restart & Clear -> Run All** (the first run puts `numpy==2.2.6` on disk).

One question, pre-committed: does the **full trimodal** TRIBE v2 pass reproduce **right-FFC (FFA) face-selectivity** on naturalistic movie clips? The pre-registered decision rule is in `.notes/plans/corticall/GATE-0.md` (D020). GO -> ROADMAP Phase 1. NO-GO -> stop, D017 fallback. AMBIGUOUS -> diagnose, don't build.

Stimuli: 8 x ~20s clips from **Tears of Steel** (live-action, CC-BY), 480p. Primary ROI: right `FFC`. Controls: `V1`, right `LO2`, `A1`. Corroborating: `PHA1-3 + VMV1-3`.

## Setup (reused verbatim from `01_setup_test.ipynb`)
### Phase 1 - install TRIBE v2 (+ shadow/numpy guards)

In [ ]:
import sys, subprocess, shutil, importlib
print('Python:', sys.version)
SRC = '/kaggle/working/tribev2_src'
shutil.rmtree('/kaggle/working/tribev2', ignore_errors=True)
shutil.rmtree(SRC, ignore_errors=True)
for _m in [m for m in list(sys.modules) if m == 'tribev2' or m.startswith('tribev2.')]:
    sys.modules.pop(_m, None)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=False)
subprocess.run(f'git clone --depth 1 https://github.com/facebookresearch/tribev2.git {SRC}',
               shell=True, check=False)
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', SRC],
                   capture_output=True, text=True)
print('tribev2 install rc =', r.returncode)
if r.returncode != 0:
    print('--- STDERR (tail) ---'); print(r.stderr[-3000:])
    print('>>> If this mentions requires-python / neuralset>=3.12, Kaggle is on Python <3.12 (G016).')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--force-reinstall', '--no-deps',
                '--no-cache-dir', '-q', 'numpy==2.2.6'], check=False)
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()
print('install step done - inspect above for resolver/version errors')
import numpy as _np
print('numpy:', _np.__version__)
import tribev2
where = getattr(tribev2, '__file__', None) or list(getattr(tribev2, '__path__', []))
print('tribev2 resolves to:', where)
assert where and 'tribev2_src' in str(where), (
    f'SHADOWED: tribev2 resolved to {where}, not {SRC}. Restart (Factory reset) and re-run.')
importlib.import_module('tribev2.demo_utils')
print('OK: tribev2.demo_utils imports.')

### Clone Corticall (tribe-bench) - brings `tribe_tools.roi_stats` etc.

In [ ]:
import os, sys, subprocess, glob
from pathlib import Path
TB_PATH = None
subprocess.run('git clone --depth 1 https://github.com/codesbydevesh/tribe-bench.git /kaggle/working/tribe-bench',
               shell=True, check=False)
if Path('/kaggle/working/tribe-bench/tribe_tools/model.py').is_file():
    TB_PATH = '/kaggle/working/tribe-bench'
if TB_PATH is None:
    for cand in glob.glob('/kaggle/input/*') + glob.glob('/kaggle/input/*/*'):
        if Path(cand, 'tribe_tools', 'model.py').is_file():
            TB_PATH = cand; break
if TB_PATH:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', TB_PATH], check=False)
    if TB_PATH not in sys.path:
        sys.path.insert(0, TB_PATH)
    print('tribe-bench found at:', TB_PATH)
else:
    print('tribe-bench NOT available - check the clone error above (repo should be public).')

### Phase 2 - environment

In [ ]:
import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB')
else:
    print('NO GPU - enable a T4 accelerator in Kaggle Settings, then re-run.')
import shutil
print('ffmpeg on PATH:', bool(shutil.which('ffmpeg')))
print('uvx on PATH   :', bool(shutil.which('uvx')), '(required for WhisperX ASR)')

### Phase 3 - HuggingFace login (gated LLaMA-3.2)

In [ ]:
import os
hf_ok = False
try:
    from huggingface_hub import login
    token = os.environ.get('HF_TOKEN', '')
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret('HF_TOKEN')
        except Exception:
            token = ''
    if token:
        login(token=token); os.environ['HF_TOKEN'] = token; hf_ok = True
        print('HF login OK')
    else:
        print('No HF_TOKEN. Set a Kaggle secret HF_TOKEN. The text (LLaMA) extractor will fail without it.')
except Exception as e:
    print('HF login error:', type(e).__name__, e)

### Phase 4 - import the wrapper

In [ ]:
WRAPPER_OK = False
try:
    from tribe_tools.model import load_model, predict_single, MODALITY_MASKS
    WRAPPER_OK = True
    print('tribe_tools.model imports: OK'); print('MODALITY_MASKS:', MODALITY_MASKS)
except Exception as e:
    print('WRAPPER IMPORT FAILED:', type(e).__name__, e)
for _m in ['tribe_tools.atlas', 'tribe_tools.cache', 'tribe_tools.viz',
           'tribe_tools.roi_stats', 'neurocheck.claims']:
    try:
        __import__(_m); print('  optional import OK:', _m)
    except Exception as e:
        print('  optional import skipped:', _m, '->', type(e).__name__, e)

## Gate 0 prep (CPU) - stimuli, covariates, atlas pre-flight. Defines `CACHE`, `FACE`, `SCENE`, ROIs.

In [ ]:
# ===== GATE 0 - PREP (CPU, pre-GPU): stimuli + covariates + atlas pre-flight =====
import subprocess, re
from pathlib import Path
import numpy as np

CACHE = Path('/kaggle/working/cache'); CACHE.mkdir(parents=True, exist_ok=True)
CLIPS = Path('/kaggle/working/clips'); CLIPS.mkdir(parents=True, exist_ok=True)

TOS_URL = 'https://download.blender.org/demo/movies/ToS/tears_of_steel_720p.mov'
tos = CACHE / 'tos.mov'
if not tos.exists() or tos.stat().st_size == 0:
    subprocess.run(['bash','-lc', f'curl -L --fail -o "{tos}" "{TOS_URL}"'], check=False)
print('ToS present:', tos.exists(), (tos.stat().st_size if tos.exists() else 0), 'bytes')

# Cut points (s), verified frame-by-frame against Tears of Steel (2026-07-24): each start sits
# inside a single sustained shot of the right content. ToS is heavily intercut, so clips are 10s
# (not 20s) to avoid straddling cuts. Still re-confirm the montage below before the GPU run.
FACE_CUTS  = [(226,'F1'), (242,'F2'), (315,'F3'), (452,'F4')]
SCENE_CUTS = [(105,'S1'), (155,'S2'), (486,'S3'), (521,'S4')]
DUR = 10

def cut(start, name):
    out = CLIPS / f'{name}.mp4'
    subprocess.run(['ffmpeg','-y','-ss',str(start),'-t',str(DUR),'-i',str(tos),
                    '-vf','scale=-2:480','-c:v','libx264','-preset','veryfast','-crf','23',
                    '-c:a','aac', str(out)], check=False, capture_output=True)
    return out

FACE  = [cut(s,n) for s,n in FACE_CUTS]
SCENE = [cut(s,n) for s,n in SCENE_CUTS]
print('clips cut:', [p.name for p in FACE+SCENE if p.exists()])

# thumbnail montage - eyeball FACE=face-dominated, SCENE=face-free, inline
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 4, figsize=(15, 5))
for ax, p in zip(axes.ravel(), FACE + SCENE):
    t = CLIPS / f'thumb_{p.stem}.png'
    subprocess.run(['ffmpeg','-y','-ss','5','-i',str(p),'-frames:v','1',str(t)],
                   check=False, capture_output=True)
    try:
        ax.imshow(plt.imread(t))
    except Exception:
        pass
    ax.set_title(p.stem); ax.axis('off')
plt.tight_layout(); plt.savefig('/kaggle/working/montage.png'); plt.show()
print('>>> CONFIRM: are the top-row (F*) clips face-dominated and bottom-row (S*) face-free? '
      'If not, edit FACE_CUTS/SCENE_CUTS and re-run this cell before the GPU loop.')

# advisory covariates (G5): luminance, audio RMS, scene-change count. Best-effort, never blocks.
def covar(path):
    d = {'luma': None, 'rms': None, 'motion': None}
    try:
        r = subprocess.run(['bash','-lc',
            f'ffmpeg -i "{path}" -vf signalstats -f null - 2>&1 | grep -o "YAVG:[0-9.]*" | tail -1'],
            capture_output=True, text=True)
        m = re.search(r'YAVG:([0-9.]+)', r.stdout);  d['luma'] = float(m.group(1)) if m else None
    except Exception: pass
    try:
        r = subprocess.run(['bash','-lc',
            f'ffmpeg -i "{path}" -af astats=metadata=1 -f null - 2>&1 | grep -o "RMS level dB:[- 0-9.]*" | tail -1'],
            capture_output=True, text=True)
        m = re.search(r'RMS level dB:\s*(-?[0-9.]+)', r.stdout);  d['rms'] = float(m.group(1)) if m else None
    except Exception: pass
    try:
        r = subprocess.run(['bash','-lc',
            f'ffmpeg -i "{path}" -vf "select=gt(scene\\,0.1),metadata=print" -f null - 2>&1 | grep -c scene_score'],
            capture_output=True, text=True)
        d['motion'] = int((r.stdout.strip() or '0').split()[0])
    except Exception: pass
    return d

print('\nCOVARIATES (advisory - reselect scene clips if FACE vs SCENE differ a lot):')
COV = {}
for grp, ps in [('FACE', FACE), ('SCENE', SCENE)]:
    for p in ps:
        COV[p.name] = {'group': grp, **covar(p)}
        print(f'  {grp:5} {p.name}: {COV[p.name]}')

# atlas pre-flight: ROIs must exist + be non-empty BEFORE any GPU spend
from tribe_tools import atlas
from tribe_tools.cache import get_cache
regions = set(atlas.list_regions())
need = {'FFC','PHA1','PHA2','PHA3','VMV1','VMV2','VMV3','V1','LO2','A1'}
missing = need - regions
assert not missing, f'atlas missing {missing}; available e.g. {sorted(regions)[:40]}'

def verts(names, hemi='both'):
    if isinstance(names, str): names = [names]
    return np.concatenate([atlas.get_vertices(n, hemi=hemi) for n in names])

FFCr    = verts('FFC', hemi='right')                          # primary = FFA
FFCbil  = verts('FFC', hemi='both')                           # secondary
SCENE_R = verts(['PHA1','PHA2','PHA3','VMV1','VMV2','VMV3'])   # corroborating scene ROI
V1v     = verts('V1'); LO2r = verts('LO2', hemi='right'); A1v = verts('A1')
for nm, v in [('FFCr',FFCr),('SCENE_R',SCENE_R),('V1',V1v),('LO2r',LO2r),('A1',A1v)]:
    assert len(v) > 0, f'empty ROI {nm}';  print(f'  {nm}: {len(v)} vertices')

cache = get_cache(CACHE / 'gate0')
print('\nPREP OK. Montage confirmed + covariates matched -> run the FULL loop.')

### Load the model (reused Phase 6; uses `CACHE` from the prep cell)

In [ ]:
import time
tribe = None; load_time = None
if WRAPPER_OK:
    try:
        t0 = time.time()
        tribe = load_model(device='cuda', cache_folder=CACHE)
        load_time = time.time() - t0
        print(f'Model loaded in {load_time:.1f}s')
    except Exception as e:
        import traceback; traceback.print_exc()
        print('MODEL LOAD FAILED:', type(e).__name__, e)
else:
    print('Skipped: wrapper not importable.')

## Gate 0 - FULL passes

In [ ]:
# ===== GATE 0 - FULL trimodal passes (GPU; ~18 min/clip @480p, HDF5-cached) =====
def run(v, mask):
    key = f"{v.resolve()}_{'full' if not mask else 'videoonly'}"
    hit = cache.load(key)
    if hit is not None:
        print('  cache hit:', v.name, ('full' if not mask else 'videoonly')); return hit
    preds, _seg = predict_single(tribe, v, features_to_mask=mask)
    cache.save(key, preds, metadata={'clip': v.name, 'mask': ('full' if not mask else 'videoonly')})
    print('  ran:', v.name, ('full' if not mask else 'videoonly'), preds.shape)
    return preds

assert tribe is not None, 'model not loaded (see the load_model cell)'
full = {v.name: run(v, None) for v in FACE + SCENE}
print('FULL passes done:', list(full))

## Gate 0 - VIDEO-ONLY passes (G4 control)

In [ ]:
# ===== GATE 0 - VIDEO-ONLY passes (G4: the speech/audio-confound control) =====
vid = {v.name: run(v, ['audio', 'text']) for v in FACE + SCENE}
print('VIDEO-ONLY passes done:', list(vid))

## Gate 0 - analysis + pre-registered verdict

In [ ]:
# ===== GATE 0 - analysis + PRE-REGISTERED decision (see .notes/plans/corticall/GATE-0.md) =====
import json
from pathlib import Path
import numpy as np
# unit-tested stats; inline fallback so this cell runs even before the repo is re-pushed
try:
    from tribe_tools.roi_stats import spatial_z, u_statistic, exact_perm_p, perm_null_deltas
except Exception:
    from itertools import combinations
    def spatial_z(preds, v):
        g = preds.mean(0) if preds.ndim == 2 else np.asarray(preds)
        sd = g.std();  return 0.0 if sd == 0 else float((g[v].mean() - g.mean()) / sd)
    def u_statistic(a, b):
        u = 0.0
        for x in a:
            for y in b: u += 1.0 if x > y else (0.5 if x == y else 0.0)
        return u
    def exact_perm_p(a, b):
        vals = list(a) + list(b); n = len(a); N = len(vals); uo = u_statistic(a, b); ge = t = 0
        for c in combinations(range(N), n):
            s = set(c); f = [vals[i] for i in range(N) if i in s]; g = [vals[i] for i in range(N) if i not in s]
            t += 1; ge += (u_statistic(f, g) >= uo - 1e-9)
        return ge / t
    def perm_null_deltas(a, b):
        vals = np.array(list(a) + list(b), float); n = len(a); N = len(vals); out = []
        for c in combinations(range(N), n):
            r = [i for i in range(N) if i not in set(c)]; out.append(vals[list(c)].mean() - vals[r].mean())
        return np.array(out)

FN = [p.name for p in FACE]; SN = [p.name for p in SCENE]

def report(passes, v, name, scene_pref=False):
    zf = [spatial_z(passes[n], v) for n in FN]; zs = [spatial_z(passes[n], v) for n in SN]
    a, b = (zs, zf) if scene_pref else (zf, zs)
    d = float(np.mean(a) - np.mean(b)); U = u_statistic(a, b); p = exact_perm_p(a, b)
    thr95 = float(np.percentile(perm_null_deltas(a, b), 95))
    print(f'{name:11} dz={d:+.3f}  U={U:.1f}/{len(zf)*len(zs)}  p={p:.4f}  (95%-null dz>{thr95:+.3f})')
    return dict(name=name, dz=d, U=float(U), n=len(zf)*len(zs), p=float(p), thr95=thr95)

print('=== FULL model ===')
FFC = report(full, FFCr, 'FFCr(FFA)'); V1r = report(full, V1v, 'V1'); LO = report(full, LO2r, 'LO2r')
SC  = report(full, SCENE_R, 'PHA+VMV', scene_pref=True); A1r = report(full, A1v, 'A1')
print('=== VIDEO-ONLY (G4) ==='); FFCv = report(vid, FFCr, 'FFCr vid')

nF = len(FN) * len(SN)
G1 = FFC['U'] >= (15 if nF == 16 else 9)                 # p<=0.029 (4v4) / 0.05 (3v3)
G2 = FFC['dz'] > FFC['thr95']
G3 = (FFC['dz'] > V1r['dz']) and (FFC['dz'] > LO['dz'])
G4 = (FFCv['dz'] > 0) and (FFCv['U'] >= (12 if nF == 16 else 7))
DD = (SC['dz'] > SC['thr95']) and (FFC['dz'] + SC['dz'] >= 0.40)   # corroborating only

if all([G1, G2, G3, G4]):
    verdict = 'GO' + (' (strong: clean double dissociation)' if DD else '')
elif (FFC['U'] <= (10 if nF == 16 else 3)) or FFC['dz'] <= 0 or (V1r['dz'] >= FFC['dz']) or (FFCv['dz'] <= 0):
    verdict = 'NO-GO -> stop; fall back to D017 static-resource paper'
else:
    verdict = 'AMBIGUOUS -> diagnose before Phase 1 (segment contrast, cross-film live-action, covariates)'

print('\nGATES  G1(dir)=%s G2(mag)=%s G3(spec)=%s G4(video-only)=%s   DD(corrob)=%s' % (G1, G2, G3, G4, DD))
print('VERDICT:', verdict)

import matplotlib.pyplot as plt
rows = [FFC, V1r, LO, SC, A1r]
plt.figure(figsize=(7, 4))
plt.bar([r['name'] for r in rows], [r['dz'] for r in rows])
plt.axhline(0, color='k', lw=.8); plt.ylabel('dz (face-pref; scene ROI = scene-pref)')
plt.title('Gate 0: ' + verdict); plt.xticks(rotation=20); plt.tight_layout()
plt.savefig('/kaggle/working/gate0_contrast.png', dpi=120); plt.show()
print('top-k ROIs (mean FACE):',
      atlas.get_topk_rois(np.stack([full[n] for n in FN]).mean(0).mean(0), k=10))

out = dict(verdict=verdict, n_face=len(FN), n_scene=len(SN),
           gates=dict(G1=bool(G1), G2=bool(G2), G3=bool(G3), G4=bool(G4), DD=bool(DD)),
           rois=dict(FFCr=FFC, V1=V1r, LO2r=LO, scene=SC, A1=A1r, FFCr_videoonly=FFCv),
           covariates=COV,
           stimulus='Tears of Steel (CC-BY 3.0, Blender Foundation), 480p, 8x20s clips')
Path('/kaggle/working/gate0_results.json').write_text(json.dumps(out, indent=2))
print('\nwrote gate0_results.json + gate0_contrast.png to /kaggle/working - DOWNLOAD before the session ends')